# config and setup

In [5]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
import pandas as pd
from pathlib import Path
from collections import defaultdict
import re

root_dir = Path("/Users/jliu/workspace/ICL/")

# load attention scores

In [13]:
class AttentionDataLoader:
    def __init__(self, data_dir, tokens_per_example=3, n_layers=6, n_heads=8):
        self.data_dir = Path(data_dir)
        self.tokens_per_example = tokens_per_example
        self.n_layers = n_layers
        self.n_heads = n_heads
        
    def parse_filename(self, filename):
        """Parse metadata from filename"""
        # Pattern: clm_noshuffle_seedbalanced_step36880_layer1_head7_k5_seq9.npz
        pattern = r'step(\d+)_layer(\d+)_head(\d+)_k(\d+)_seq(\d+)\.npz'
        match = re.match(r'.*' + pattern, filename)
        
        if match:
            return {
                'step': int(match.group(1)),
                'layer': int(match.group(2)),
                'head': int(match.group(3)),
                'n_shots': int(match.group(4)),
                'seq_id': int(match.group(5))
            }
        return None
    
    def load_attention_files(self, step=None, step_range=None, max_sequences=None, 
                           sequence_ids=None, seq_id_range=None, n_shots_filter=None,
                           require_complete=True, sampling_strategy='first'):
        """
        Load and organize attention data from .npz files
        
        Args:
            step: Specific training step to load
            step_range: Tuple (min_step, max_step) to load range of steps
            max_sequences: Maximum number of sequences to load
            sequence_ids: List of specific sequence IDs to load
            seq_id_range: Tuple (min_seq, max_seq) for sequence ID range
            n_shots_filter: List of shot counts to include (e.g., [3, 5])
            require_complete: Only include sequences with all layer/head combinations
            sampling_strategy: 'first', 'random', 'stratified' for sequence selection
        """
        files_data = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: None)))
        sequence_info = {}
        file_inventory = defaultdict(lambda: defaultdict(lambda: defaultdict(bool)))
        
        # Get all .npz files
        npz_files = list(self.data_dir.glob("*.npz"))
        print(f"Found {len(npz_files)} .npz files")
        
        # First pass: inventory all files and load data
        for file_path in npz_files:
            metadata = self.parse_filename(file_path.name)
            if not metadata:
                print(f"Warning: Could not parse filename {file_path.name}")
                continue
                
            # Filter by step or step range
            if step is not None and metadata['step'] != step:
                continue
            if step_range is not None:
                min_step, max_step = step_range
                if metadata['step'] < min_step or metadata['step'] > max_step:
                    continue
            
            # Filter by sequence ID
            if sequence_ids is not None and metadata['seq_id'] not in sequence_ids:
                continue
            if seq_id_range is not None:
                min_seq, max_seq = seq_id_range
                if metadata['seq_id'] < min_seq or metadata['seq_id'] > max_seq:
                    continue
            
            seq_id = metadata['seq_id']
            layer_id = metadata['layer']
            head_id = metadata['head']
            
            
            try:
                # Load attention matrix
                data = np.load(file_path)
                attention_matrix = data['attention_matrix']  # Adjust key name if different
                
                # Store data organized by sequence -> layer -> head
                files_data[seq_id][layer_id][head_id] = attention_matrix
                file_inventory[seq_id][layer_id][head_id] = True
                
                # Store sequence metadata (should be consistent across files for same sequence)
                
                sequence_info[seq_id] = {
                    'n_shots': metadata['n_shots'],
                    'step': metadata['step']
                }
                
                        
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
                continue
        
        # Filter by n_shots after loading metadata
        if n_shots_filter is not None:
            filtered_sequences = {seq_id: info for seq_id, info in sequence_info.items() 
                                if info['n_shots'] in n_shots_filter}
            # Remove sequences that don't match n_shots filter
            sequences_to_remove = set(sequence_info.keys()) - set(filtered_sequences.keys())
            for seq_id in sequences_to_remove:
                if seq_id in files_data:
                    del files_data[seq_id]
                if seq_id in file_inventory:
                    del file_inventory[seq_id]
            sequence_info = filtered_sequences
        
        print(f"Successfully loaded data for {len(files_data)} sequences")
        
        # Second pass: check completeness and filter sequences
        complete_sequences = {}
        incomplete_sequences = {}
        
        for seq_id in files_data:
            missing_combinations = []
            
            # Check if we have all expected layer/head combinations
            for layer_idx in range(self.n_layers):
                for head_idx in range(self.n_heads):
                    if not file_inventory[seq_id][layer_idx][head_idx]:
                        missing_combinations.append((layer_idx, head_idx))
            
            if missing_combinations:
                incomplete_sequences[seq_id] = {
                    'data': files_data[seq_id],
                    'missing': missing_combinations,
                    'coverage': (self.n_layers * self.n_heads - len(missing_combinations)) / (self.n_layers * self.n_heads)
                }
                if not require_complete:
                    # Fill missing combinations with None
                    for layer_idx, head_idx in missing_combinations:
                        files_data[seq_id][layer_idx][head_idx] = None
            else:
                complete_sequences[seq_id] = files_data[seq_id]
        
        
        if incomplete_sequences and not require_complete:
            print("Incomplete sequence details:")
            for seq_id, info in list(incomplete_sequences.items())[:5]:  # Show first 5
                coverage = info['coverage'] * 100
                print(f"  Seq {seq_id}: {coverage:.1f}% coverage, missing {len(info['missing'])} combinations")
        
        # Choose which sequences to return
        if require_complete:
            final_data = complete_sequences
            final_info = {seq_id: sequence_info[seq_id] for seq_id in complete_sequences}
        else:
            final_data = files_data
            final_info = sequence_info
        
        # Apply sampling strategy and limit sequences
        if max_sequences and len(final_data) > max_sequences:
            sequence_ids_list = list(final_data.keys())
            
            if sampling_strategy == 'first':
                selected_ids = sequence_ids_list[:max_sequences]
            elif sampling_strategy == 'random':
                import random
                selected_ids = random.sample(sequence_ids_list, max_sequences)
            elif sampling_strategy == 'stratified':
                # Stratify by n_shots
                shots_groups = defaultdict(list)
                for seq_id in sequence_ids_list:
                    n_shots = final_info[seq_id]['n_shots']
                    shots_groups[n_shots].append(seq_id)
                
                selected_ids = []
                shots_per_group = max_sequences // len(shots_groups)
                remaining = max_sequences % len(shots_groups)
                
                for i, (n_shots, seq_ids) in enumerate(sorted(shots_groups.items())):
                    group_size = shots_per_group + (1 if i < remaining else 0)
                    if len(seq_ids) > group_size:
                        import random
                        selected_ids.extend(random.sample(seq_ids, group_size))
                    else:
                        selected_ids.extend(seq_ids)
            else:
                selected_ids = sequence_ids_list[:max_sequences]
                
            final_data = {seq_id: final_data[seq_id] for seq_id in selected_ids}
            final_info = {seq_id: final_info[seq_id] for seq_id in selected_ids}
            print(f"Applied {sampling_strategy} sampling to select {len(selected_ids)} sequences")
        
        return final_data, final_info, {'complete': complete_sequences, 'incomplete': incomplete_sequences}
    
    def convert_to_analysis_format(self, files_data, sequence_info):
        """Convert loaded data to format expected by AttentionPatternExtractor"""
        attention_data = []
        sequence_metadata = []
        
        for seq_id, seq_data in files_data.items():
            # Initialize attention structure for this sequence
            seq_attention = {}
            
            for layer_idx in range(self.n_layers):
                seq_attention[layer_idx] = {}
                
                for head_idx in range(self.n_heads):
                    # Get attention matrix (could be None if missing)
                    attention_matrix = seq_data.get(layer_idx, {}).get(head_idx, None)
                    seq_attention[layer_idx][head_idx] = attention_matrix
                    
            # Create sequence metadata
            n_shots = sequence_info[seq_id]['n_shots']
            context_boundaries = self.compute_context_boundaries(n_shots)
            query_position = self.compute_query_position(n_shots)
            
            metadata = {
                'seq_id': seq_id,
                'context_boundaries': context_boundaries,
                'query_position': query_position,
                'rule_complexity': self.infer_rule_complexity(n_shots),
                'n_shots': n_shots,
                'step': sequence_info[seq_id]['step']
            }
            
            attention_data.append(seq_attention)
            sequence_metadata.append(metadata)
        
        return attention_data, sequence_metadata

    def compute_context_boundaries(self, n_shots):
        """Compute context example boundaries"""
        boundaries = []
        for i in range(n_shots):
            start = i * self.tokens_per_example
            end = (i + 1) * self.tokens_per_example
            boundaries.append((start, end))
        return boundaries

    def compute_query_position(self, n_shots):
        """Compute query start position"""
        return n_shots * self.tokens_per_example

    def infer_rule_complexity(self, n_shots):
        """Simple heuristic: more shots = higher complexity"""
        if n_shots <= 2:
            return 1  # Simple
        elif n_shots <= 4:
            return 2  # Medium
        else:
            return 3  # Complex
            
    def validate_data(self, attention_data, sequence_metadata):
        """Enhanced validation of loaded data quality"""
        print("\n" + "="*50)
        print("DATA VALIDATION SUMMARY")
        print("="*50)
        print(f"Total sequences: {len(attention_data)}")
        
        if not attention_data:
            print("No data to validate!")
            return 0
        
        # Check completeness and collect statistics
        complete_sequences = 0
        total_matrices = 0
        missing_matrices = 0
        matrix_shapes = defaultdict(int)
        
        for i, seq_attention in enumerate(attention_data):
            seq_complete = True
            seq_total = 0
            seq_missing = 0
            
            for layer_idx in range(self.n_layers):
                for head_idx in range(self.n_heads):
                    seq_total += 1
                    total_matrices += 1
                    
                    attention_matrix = seq_attention.get(layer_idx, {}).get(head_idx, None)
                    
                    if attention_matrix is None:
                        seq_missing += 1
                        missing_matrices += 1
                        seq_complete = False
                    else:
                        # Record shape
                        shape = attention_matrix.shape
                        matrix_shapes[shape] += 1
            
            if seq_complete:
                complete_sequences += 1
            elif seq_missing > 0:
                coverage = (seq_total - seq_missing) / seq_total * 100
                print(f"  Seq {sequence_metadata[i]['seq_id']}: {coverage:.1f}% coverage ({seq_missing} missing)")

        print(f"Complete sequences: {complete_sequences}/{len(attention_data)} ({complete_sequences/len(attention_data)*100:.1f}%)")
        print(f"Total attention matrices: {total_matrices - missing_matrices}/{total_matrices}")
        
        if missing_matrices > 0:
            print(f"Missing matrices: {missing_matrices} ({missing_matrices/total_matrices*100:.1f}%)")
        
        # Show matrix shapes
        print(f"\nAttention matrix shapes found:")
        for shape, count in sorted(matrix_shapes.items()):
            print(f"  {shape}: {count} matrices")
        
        # Show shots distribution
        shots_dist = defaultdict(int)
        steps_found = set()
        for metadata in sequence_metadata:
            shots_dist[metadata['n_shots']] += 1
            steps_found.add(metadata['step'])
            
        print(f"\nShots distribution: {dict(sorted(shots_dist.items()))}")
        print(f"Training steps found: {sorted(steps_found)}")
        
        # Sample attention matrix info
        if complete_sequences > 0:
            # Find first complete sequence
            for i, seq_attention in enumerate(attention_data):
                sample_matrix = None
                for layer_idx in range(self.n_layers):
                    for head_idx in range(self.n_heads):
                        matrix = seq_attention.get(layer_idx, {}).get(head_idx, None)
                        if matrix is not None:
                            sample_matrix = matrix
                            print(f"\nSample matrix (Seq {sequence_metadata[i]['seq_id']}, L{layer_idx}H{head_idx}):")
                            print(f"  Shape: {matrix.shape}")
                            print(f"  Min/Max: {matrix.min():.4f}/{matrix.max():.4f}")
                            print(f"  Sum: {matrix.sum():.4f}")
                            break
                    if sample_matrix is not None:
                        break
                if sample_matrix is not None:
                    break
        
        print("="*50)
        return complete_sequences
    
    def get_data_statistics(self, completeness_info):
        """Get detailed statistics about data completeness"""
        complete = completeness_info['complete']
        incomplete = completeness_info['incomplete']
        
        stats = {
            'total_sequences': len(complete) + len(incomplete),
            'complete_sequences': len(complete),
            'incomplete_sequences': len(incomplete),
            'completion_rate': len(complete) / (len(complete) + len(incomplete)) if (len(complete) + len(incomplete)) > 0 else 0
        }
        
        if incomplete:
            coverages = [info['coverage'] for info in incomplete.values()]
            stats['incomplete_coverage_stats'] = {
                'mean': np.mean(coverages),
                'min': np.min(coverages),
                'max': np.max(coverages)
            }
        
        return stats

In [10]:
import re
import numpy as np
from pathlib import Path
from collections import defaultdict

class AttentionDataLoaderBatch:
    """Batch generator for attention matrices to feed streaming Phase-1 pipeline"""
    def __init__(self, data_dir, n_layers=6, n_heads=8, tokens_per_example=9, batch_size=50):
        self.data_dir = Path(data_dir)
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.tokens_per_example = tokens_per_example
        self.batch_size = batch_size

    def parse_filename(self, filename):
        """Parse metadata from filename"""
        pattern = r'step(\d+)_layer(\d+)_head(\d+)_k(\d+)_seq(\d+)\.npz'
        match = re.match(r'.*' + pattern, filename)
        if match:
            return {
                'step': int(match.group(1)),
                'layer': int(match.group(2)),
                'head': int(match.group(3)),
                'n_shots': int(match.group(4)),
                'seq_id': int(match.group(5))
            }
        return None

    def compute_context_boundaries(self, n_shots):
        return [(i * self.tokens_per_example, (i + 1) * self.tokens_per_example) for i in range(n_shots)]

    def compute_query_position(self, n_shots):
        return n_shots * self.tokens_per_example

    def infer_rule_complexity(self, n_shots):
        if n_shots <= 2: return 1
        if n_shots <= 4: return 2
        return 3

    def batch_generator(self):
        """Yield batches of files_data and sequence_metadata"""
        seq_data_store = defaultdict(lambda: defaultdict(dict))
        seq_metadata_store = {}
        batch_counter = 0

        npz_files = list(self.data_dir.glob("*.npz"))
        for file_path in npz_files:
            metadata = self.parse_filename(file_path.name)
            if not metadata:
                continue

            seq_id = metadata['seq_id']
            layer = metadata['layer']
            head = metadata['head']

            # Load attention matrix
            try:
                data = np.load(file_path)
                attn_matrix = data['attention_matrix']
            except:
                continue

            seq_data_store[seq_id][layer][head] = attn_matrix

            # Store sequence-level metadata once
            if seq_id not in seq_metadata_store:
                n_shots = metadata['n_shots']
                seq_metadata_store[seq_id] = {
                    'n_shots': n_shots,
                    'context_boundaries': self.compute_context_boundaries(n_shots),
                    'query_position': self.compute_query_position(n_shots),
                    'rule_complexity': self.infer_rule_complexity(n_shots),
                    'step': metadata['step']
                }

            # Yield batch if batch size reached
            batch_counter += 1
            if batch_counter >= self.batch_size:
                yield dict(seq_data_store), dict(seq_metadata_store)
                seq_data_store.clear()
                seq_metadata_store.clear()
                batch_counter = 0

        # Yield any remaining sequences
        if seq_data_store:
            yield dict(seq_data_store), dict(seq_metadata_store)


# extract attention patterns

In [ ]:
# new version
class AttentionPatternExtractor:
    def __init__(self, config, missing_data_strategy='skip'):
        self.config = config
        self.missing_data_strategy = missing_data_strategy  # 'skip', 'nan', 'zero'
        
    def _handle_missing_data(self, value, default_value=0.0):
        """Handle missing data based on strategy"""
        if self.missing_data_strategy == 'skip':
            return None  # Will be filtered out later
        elif self.missing_data_strategy == 'nan':
            return np.nan
        elif self.missing_data_strategy == 'zero':
            return default_value
        else:
            return value
    
    def compute_attention_entropy(self, attention_matrix):
        """Compute entropy for each query position with None handling"""
        if attention_matrix is None:
            return self._handle_missing_data(None, default_value=0.0)
            
        # Avoid log(0) by adding small epsilon
        attention_matrix = attention_matrix + 1e-8
        entropies = []
        
        try:
            for i in range(attention_matrix.shape[0]):
                row_entropy = entropy(attention_matrix[i, :])
                entropies.append(row_entropy)
            return np.mean(entropies)
        except Exception as e:
            print(f"Error computing entropy: {e}")
            return self._handle_missing_data(None, default_value=0.0)
    
    def compute_locality_score(self, attention_matrix, context_boundaries):
        """Measure how much attention stays within examples with None handling"""
        if attention_matrix is None:
            return self._handle_missing_data(None, default_value=0.0)
            
        if not context_boundaries:
            return 0.0
            
        try:
            intra_example_attention = 0.0
            total_attention = 0.0
            
            for start, end in context_boundaries:
                # Ensure boundaries are within matrix dimensions
                if start >= attention_matrix.shape[0] or end > attention_matrix.shape[1]:
                    continue
                    
                # Attention within this example
                example_block = attention_matrix[start:end, start:end]
                intra_example_attention += np.sum(example_block)
                
                # Total attention from this example
                example_queries = attention_matrix[start:end, :]
                total_attention += np.sum(example_queries)
            
            return intra_example_attention / (total_attention + 1e-8)
            
        except Exception as e:
            print(f"Error computing locality score: {e}")
            return self._handle_missing_data(None, default_value=0.0)
    
    def compute_cross_example_attention(self, attention_matrix, context_boundaries):
        """Measure attention between different examples with None handling"""
        if attention_matrix is None:
            return self._handle_missing_data(None, default_value=0.0)
            
        if len(context_boundaries) < 2:
            return 0.0
            
        try:
            cross_attention = 0.0
            total_attention = 0.0
            
            for i, (start1, end1) in enumerate(context_boundaries):
                # Ensure boundaries are within matrix dimensions
                if start1 >= attention_matrix.shape[0] or end1 > attention_matrix.shape[1]:
                    continue
                    
                for j, (start2, end2) in enumerate(context_boundaries):
                    if i != j and start2 < attention_matrix.shape[1] and end2 <= attention_matrix.shape[1]:
                        cross_block = attention_matrix[start1:end1, start2:end2]
                        cross_attention += np.sum(cross_block)
                
                example_queries = attention_matrix[start1:end1, :]
                total_attention += np.sum(example_queries)
            
            return cross_attention / (total_attention + 1e-8)
            
        except Exception as e:
            print(f"Error computing cross-example attention: {e}")
            return self._handle_missing_data(None, default_value=0.0)
    
    def compute_icl_patterns(self, attention_matrix, context_boundaries, query_position):
        """Analyze ICL-specific attention patterns with None handling"""
        if attention_matrix is None:
            return {
                'context_to_query': self._handle_missing_data(None, default_value=0.0),
                'query_to_context': self._handle_missing_data(None, default_value=0.0)
            }
            
        if query_position is None or not context_boundaries:
            return {'context_to_query': 0.0, 'query_to_context': 0.0}
        
        try:
            # Ensure query_position is within matrix dimensions
            if query_position >= attention_matrix.shape[0]:
                return {'context_to_query': 0.0, 'query_to_context': 0.0}
            
            # Context → Query attention
            context_to_query = 0.0
            for start, end in context_boundaries:
                if start < attention_matrix.shape[1] and end <= attention_matrix.shape[1]:
                    context_to_query += np.sum(attention_matrix[query_position, start:end])
            
            # Query → Context attention (for causal models, this should be 0)
            query_to_context = 0.0
            for start, end in context_boundaries:
                if (query_position < start and 
                    start < attention_matrix.shape[0] and 
                    end <= attention_matrix.shape[0]):
                    query_to_context += np.sum(attention_matrix[start:end, query_position])
            
            return {
                'context_to_query': context_to_query,
                'query_to_context': query_to_context
            }
            
        except Exception as e:
            print(f"Error computing ICL patterns: {e}")
            return {
                'context_to_query': self._handle_missing_data(None, default_value=0.0),
                'query_to_context': self._handle_missing_data(None, default_value=0.0)
            }

    def extract_head_features(self, attention_data, sequence_metadata):
        """Extract features for all heads across all sequences with robust None handling"""
        features = []
        head_identifiers = []
        skipped_count = 0
        
        for seq_idx, (attention_seq, metadata) in enumerate(zip(attention_data, sequence_metadata)):
            context_boundaries = metadata.get('context_boundaries', [])
            query_position = metadata.get('query_position', None)
            rule_complexity = metadata.get('rule_complexity', 1)
            
            for layer_idx in range(self.config.n_layers):
                for head_idx in range(self.config.n_heads):
                    attention_matrix = attention_seq.get(layer_idx, {}).get(head_idx, None)
                    
                    # Compute all features with None handling
                    entropy_score = self.compute_attention_entropy(attention_matrix)
                    locality_score = self.compute_locality_score(attention_matrix, context_boundaries)
                    cross_example_score = self.compute_cross_example_attention(attention_matrix, context_boundaries)
                    icl_patterns = self.compute_icl_patterns(attention_matrix, context_boundaries, query_position)
                    
                    # Prepare feature vector
                    feature_vector = [
                        entropy_score,
                        locality_score, 
                        cross_example_score,
                        icl_patterns['context_to_query'],
                        icl_patterns['query_to_context'],
                        rule_complexity,
                        layer_idx,
                        head_idx
                    ]
                    
                    # Handle missing data based on strategy
                    if self.missing_data_strategy == 'skip':
                        # Skip if any feature is None
                        if any(f is None for f in feature_vector[:5]):  # Skip first 5 feature values
                            skipped_count += 1
                            continue
                    
                    features.append(feature_vector)
                    head_identifiers.append((seq_idx, layer_idx, head_idx))
        
        if skipped_count > 0:
            print(f"Skipped {skipped_count} head features due to missing data")
            
        features_array = np.array(features)
        
        # Additional validation for NaN values
        if np.isnan(features_array).any():
            nan_count = np.isnan(features_array).sum()
            print(f"Warning: {nan_count} NaN values found in features")
            
            if self.missing_data_strategy == 'nan':
                print("NaN values preserved as per missing_data_strategy='nan'")
            else:
                print("Consider using missing_data_strategy='nan' if NaN values are expected")
        
        return features_array, head_identifiers
    
    def validate_attention_data(self, attention_data, sequence_metadata):
        """Validate attention data structure and report missing data statistics"""
        print("\n" + "="*50)
        print("ATTENTION DATA VALIDATION")
        print("="*50)
        
        total_matrices = 0
        missing_matrices = 0
        sequences_with_missing = 0
        
        for seq_idx, (attention_seq, metadata) in enumerate(zip(attention_data, sequence_metadata)):
            seq_missing = 0
            
            for layer_idx in range(self.config.n_layers):
                for head_idx in range(self.config.n_heads):
                    total_matrices += 1
                    attention_matrix = attention_seq.get(layer_idx, {}).get(head_idx, None)
                    
                    if attention_matrix is None:
                        missing_matrices += 1
                        seq_missing += 1
                    else:
                        # Validate matrix properties
                        if not isinstance(attention_matrix, np.ndarray):
                            print(f"Warning: Non-numpy array found at seq {seq_idx}, L{layer_idx}H{head_idx}")
                        elif attention_matrix.size == 0:
                            print(f"Warning: Empty matrix found at seq {seq_idx}, L{layer_idx}H{head_idx}")
                        elif np.isnan(attention_matrix).any():
                            print(f"Warning: NaN values in matrix at seq {seq_idx}, L{layer_idx}H{head_idx}")
            
            if seq_missing > 0:
                sequences_with_missing += 1
        
        print(f"Total attention matrices expected: {total_matrices}")
        print(f"Missing matrices: {missing_matrices} ({missing_matrices/total_matrices*100:.1f}%)")
        print(f"Sequences with missing data: {sequences_with_missing}/{len(attention_data)}")
        print(f"Missing data strategy: {self.missing_data_strategy}")
        
        # Estimate impact on feature extraction
        if missing_matrices > 0:
            if self.missing_data_strategy == 'skip':
                print(f"Estimated features to be skipped: {missing_matrices}")
            elif self.missing_data_strategy == 'nan':
                print(f"Features with NaN values: {missing_matrices}")
            elif self.missing_data_strategy == 'zero':
                print(f"Features with default values: {missing_matrices}")
        
        print("="*50)
        
        return {
            'total_matrices': total_matrices,
            'missing_matrices': missing_matrices,
            'sequences_with_missing': sequences_with_missing,
            'missing_rate': missing_matrices / total_matrices if total_matrices > 0 else 0
        }

In [11]:
# batch loader

import numpy as np
from collections import defaultdict
from scipy.stats import entropy
from sklearn.cluster import MiniBatchKMeans

class AttentionPatternExtractorStreaming:
    """Streaming, batch-safe feature extractor for Phase-1 analysis"""
    def __init__(self, config, missing_data_strategy='skip'):
        self.config = config
        self.missing_data_strategy = missing_data_strategy  # 'skip', 'nan', 'zero'

    def _handle_missing_data(self, value, default_value=0.0):
        if self.missing_data_strategy == 'skip':
            return None
        elif self.missing_data_strategy == 'nan':
            return np.nan
        elif self.missing_data_strategy == 'zero':
            return default_value
        else:
            return value

    def compute_attention_entropy(self, attention_matrix):
        if attention_matrix is None:
            return self._handle_missing_data(None, 0.0)
        attention_matrix += 1e-8
        try:
            return float(np.mean([entropy(row) for row in attention_matrix]))
        except:
            return self._handle_missing_data(None, 0.0)

    def compute_locality_score(self, attention_matrix, context_boundaries):
        if attention_matrix is None:
            return self._handle_missing_data(None, 0.0)
        try:
            intra_attention = 0.0
            total_attention = 0.0
            for start, end in context_boundaries:
                if start >= attention_matrix.shape[0] or end > attention_matrix.shape[1]:
                    continue
                block = attention_matrix[start:end, start:end]
                intra_attention += np.sum(block)
                total_attention += np.sum(attention_matrix[start:end, :])
            return intra_attention / (total_attention + 1e-8)
        except:
            return self._handle_missing_data(None, 0.0)

    def compute_cross_example_attention(self, attention_matrix, context_boundaries):
        if attention_matrix is None:
            return self._handle_missing_data(None, 0.0)
        try:
            cross_attention = 0.0
            total_attention = 0.0
            for i, (start1, end1) in enumerate(context_boundaries):
                if start1 >= attention_matrix.shape[0] or end1 > attention_matrix.shape[1]:
                    continue
                for j, (start2, end2) in enumerate(context_boundaries):
                    if i != j and start2 < attention_matrix.shape[1] and end2 <= attention_matrix.shape[1]:
                        cross_attention += np.sum(attention_matrix[start1:end1, start2:end2])
                total_attention += np.sum(attention_matrix[start1:end1, :])
            return cross_attention / (total_attention + 1e-8)
        except:
            return self._handle_missing_data(None, 0.0)

    def compute_icl_patterns(self, attention_matrix, context_boundaries, query_position):
        if attention_matrix is None:
            return {'context_to_query': self._handle_missing_data(None, 0.0),
                    'query_to_context': self._handle_missing_data(None, 0.0)}
        if query_position is None or not context_boundaries:
            return {'context_to_query': 0.0, 'query_to_context': 0.0}
        try:
            context_to_query = sum(
                np.sum(attention_matrix[query_position, start:end])
                for start, end in context_boundaries
                if start < attention_matrix.shape[1] and end <= attention_matrix.shape[1]
            )
            query_to_context = 0.0
            for start, end in context_boundaries:
                if query_position < start and start < attention_matrix.shape[0] and end <= attention_matrix.shape[0]:
                    query_to_context += np.sum(attention_matrix[start:end, query_position])
            return {'context_to_query': context_to_query, 'query_to_context': query_to_context}
        except:
            return {'context_to_query': self._handle_missing_data(None, 0.0),
                    'query_to_context': self._handle_missing_data(None, 0.0)}

    def extract_head_features_streaming(self, files_data, sequence_metadata):
        """Generator that yields features per head, per sequence"""
        for seq_id, seq_attention in files_data.items():
            context_boundaries = sequence_metadata[seq_id]['context_boundaries']
            query_position = sequence_metadata[seq_id]['query_position']
            rule_complexity = sequence_metadata[seq_id]['rule_complexity']

            for layer_idx in range(self.config.n_layers):
                for head_idx in range(self.config.n_heads):
                    attention_matrix = seq_attention.get(layer_idx, {}).get(head_idx, None)

                    entropy_score = self.compute_attention_entropy(attention_matrix)
                    locality_score = self.compute_locality_score(attention_matrix, context_boundaries)
                    cross_score = self.compute_cross_example_attention(attention_matrix, context_boundaries)
                    icl_patterns = self.compute_icl_patterns(attention_matrix, context_boundaries, query_position)

                    feature_vector = [
                        entropy_score,
                        locality_score,
                        cross_score,
                        icl_patterns['context_to_query'],
                        icl_patterns['query_to_context'],
                        rule_complexity,
                        layer_idx,
                        head_idx
                    ]

                    if self.missing_data_strategy == 'skip' and any(f is None for f in feature_vector[:5]):
                        continue

                    yield np.array(feature_vector, dtype=np.float32), (seq_id, layer_idx, head_idx)
                    # Discard matrix
                    del attention_matrix



class Phase1AnalysisPipelineBatch:
    """Phase-1 pipeline with streaming feature extraction and incremental clustering"""
    def __init__(self, config, missing_data_strategy='skip', batch_size=50):
        self.config = config
        self.extractor = AttentionPatternExtractorStreaming(config, missing_data_strategy)
        self.batch_size = batch_size

    def run_phase1_batches(self, batch_generator):
        """Run Phase-1 analysis on batches"""
        all_features = []
        all_head_ids = []

        # MiniBatchKMeans for incremental clustering
        clusterer = MiniBatchKMeans(n_clusters=self.config.n_clusters, batch_size=self.batch_size)

        for files_data, seq_meta in batch_generator:
            batch_features = []
            batch_ids = []

            for feature_vector, head_id in self.extractor.extract_head_features_streaming(files_data, seq_meta):
                batch_features.append(feature_vector)
                batch_ids.append(head_id)

            if batch_features:
                batch_features_np = np.stack(batch_features)
                all_features.append(batch_features_np)
                all_head_ids.extend(batch_ids)

                # Incremental clustering
                clusterer.partial_fit(batch_features_np)

            # Free memory
            del batch_features_np, batch_features, files_data, seq_meta

        if all_features:
            features_array = np.vstack(all_features)
        else:
            features_array = np.zeros((0, 8), dtype=np.float32)

        return {
            'features_array': features_array,
            'head_identifiers': all_head_ids,
            'clusterer_model': clusterer
        }


# analyze specific layers

In [12]:
import numpy as np
from scipy.stats import f_oneway, spearmanr, chi2_contingency, sem
from collections import defaultdict

# ------------------------------
# 1. LayerSpecializationAnalyzer
# ------------------------------
class LayerSpecializationAnalyzer:
    def __init__(self, n_layers, feature_names=None):
        self.n_layers = n_layers
        self.feature_names = feature_names or []

    def compute_layer_stats(self, features_array, head_identifiers):
        """
        Compute per-layer mean, CI, ANOVA/Kruskal-Wallis, effect sizes
        """
        n_features = features_array.shape[1]
        layer_groups = defaultdict(list)

        # Group features by layer
        for feat_idx, (seq_idx, layer_idx, head_idx) in enumerate(head_identifiers):
            layer_groups[layer_idx].append(features_array[feat_idx, :])

        layer_stats = {
            'layer_means': {},
            'layer_cis': {},
            'anova_results': {},
            'effect_sizes': {}
        }

        # Compute per-feature
        for f_idx in range(n_features):
            # Collect per-layer vectors
            per_layer_vectors = [np.array(layer_groups[l])[:, f_idx] for l in range(self.n_layers)]
            # Compute mean ± CI
            layer_stats['layer_means'][f_idx] = [np.mean(v) for v in per_layer_vectors]
            layer_stats['layer_cis'][f_idx] = [1.96 * sem(v) if len(v) > 1 else 0.0 for v in per_layer_vectors]
            # ANOVA across layers
            layer_stats['anova_results'][f_idx] = f_oneway(*per_layer_vectors)
            # η² effect size: SS_between / SS_total
            grand_mean = np.mean(np.concatenate(per_layer_vectors))
            ss_between = sum(len(v) * (np.mean(v) - grand_mean)**2 for v in per_layer_vectors)
            ss_total = sum(sum((x - grand_mean)**2 for x in v) for v in per_layer_vectors)
            layer_stats['effect_sizes'][f_idx] = ss_between / ss_total if ss_total > 0 else 0.0

        return layer_stats



# analyze k-shot differences

In [13]:

# ------------------------------
# 2. ComplexityTrendAnalyzer
# ------------------------------
class ComplexityTrendAnalyzer:
    def compute_nshot_trends(self, features_array, sequence_metadata):
        """
        Compute trends of features vs n_shots
        """
        n_features = features_array.shape[1]
        n_shots_list = [meta['n_shots'] for meta in sequence_metadata]
        trend_stats = {
            'linear_slopes': {},
            'r_squared': {},
            'spearman_rho': {},
            'effect_sizes': {}
        }

        for f_idx in range(n_features):
            y = features_array[:, f_idx]
            x = np.array(n_shots_list)

            # Linear slope
            slope = np.polyfit(x, y, 1)[0]
            trend_stats['linear_slopes'][f_idx] = slope

            # R²
            y_pred = slope * x + np.mean(y - slope * x)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            trend_stats['r_squared'][f_idx] = r2

            # Spearman rho
            rho, _ = spearmanr(x, y)
            trend_stats['spearman_rho'][f_idx] = rho

            # Cohen's d (low vs high shots)
            low = y[np.array(x) <= 2]
            high = y[np.array(x) >= 3]
            pooled_sd = np.sqrt((np.var(low, ddof=1) + np.var(high, ddof=1)) / 2) if len(low) > 1 and len(high) > 1 else 0.0
            d = (np.mean(high) - np.mean(low)) / pooled_sd if pooled_sd > 0 else 0.0
            trend_stats['effect_sizes'][f_idx] = d

        return trend_stats



# cluster layers

In [14]:

# ------------------------------
# 3. ClusterLayerAssociationAnalyzer
# ------------------------------
class ClusterLayerAssociationAnalyzer:
    def __init__(self, n_layers):
        self.n_layers = n_layers

    def compute_layer_enrichment(self, clusters, head_identifiers):
        """
        Compute cluster vs layer association
        """
        layers = [hid[1] for hid in head_identifiers]
        contingency_table = defaultdict(lambda: defaultdict(int))

        # Build contingency table
        for c, l in zip(clusters, layers):
            contingency_table[c][l] += 1

        # Convert to 2D array
        cluster_ids = sorted(contingency_table.keys())
        layer_ids = list(range(self.n_layers))
        table = np.array([[contingency_table[c][l] for l in layer_ids] for c in cluster_ids])

        chi2_stat, p_value, dof, expected = chi2_contingency(table)
        std_residuals = (table - expected) / np.sqrt(expected)

        return {
            'chi2_stat': chi2_stat,
            'p_value': p_value,
            'std_residuals': std_residuals
        }



# ------------------------------
# 4. ClusterPerformanceMapper
# ------------------------------
class ClusterPerformanceMapper:
    def __init__(self, outcome_labels):
        self.outcome_labels = outcome_labels  # aligned with sequences

    def map_clusters_to_performance(self, clusters, features_array, head_identifiers, sequence_metadata):
        """
        Map clusters to Mem / ID / OOD outcomes
        """
        # Map seq_idx → cluster
        seq_to_cluster = {}
        for (seq_idx, layer_idx, head_idx), cluster in zip(head_identifiers, clusters):
            seq_to_cluster[seq_idx] = cluster

        perf_summary = defaultdict(lambda: defaultdict(list))  # cluster → outcome → features

        for seq_idx, meta in enumerate(sequence_metadata):
            outcome = self.outcome_labels[seq_idx]
            cluster = seq_to_cluster.get(seq_idx, None)
            if cluster is not None:
                perf_summary[cluster][outcome].append(features_array[seq_idx, :])

        # Compute cluster-feature means ± CI
        cluster_feature_means = {}
        contrasts = {}
        for cluster, outcome_dict in perf_summary.items():
            cluster_feature_means[cluster] = {}
            for outcome, feats in outcome_dict.items():
                feats_array = np.array(feats)
                cluster_feature_means[cluster][outcome] = {
                    'mean': np.mean(feats_array, axis=0),
                    'ci': 1.96 * sem(feats_array, axis=0) if feats_array.shape[0] > 1 else np.zeros(feats_array.shape[1])
                }
            # Optional: compute contrasts effect sizes between outcomes
            # Placeholder: compute Cohen's d between Mem vs ID vs OOD
        return {
            'cluster_feature_means': cluster_feature_means,
            'contrasts': contrasts,
            'performance_summary': perf_summary
        }


# cluster heads

In [15]:
# new code

import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from collections import defaultdict


class HeadClusterer:
    def __init__(self, config, missing_data_strategy="skip", imputation_strategy="mean", n_clusters=6, random_state=42):
        """
        Initialize the head clustering module.
        Args:
            config: AnalysisConfig object with n_layers, n_heads, etc.
            missing_data_strategy: 'skip', 'zero', or 'nan'
            imputation_strategy: Used only if missing_data_strategy='nan' ('mean', 'median', 'most_frequent', 'constant')
            n_clusters: Number of clusters for KMeans
            random_state: For reproducibility
        """
        self.config = config
        self.missing_data_strategy = missing_data_strategy
        self.imputation_strategy = imputation_strategy
        self.n_clusters = n_clusters
        self.random_state = random_state

    def _preprocess_features(self, features, head_identifiers):
        """
        Handle missing data and standardize features before clustering.
        Returns cleaned_features, cleaned_identifiers, skipped_heads, imputer (if used)
        """
        skipped_heads = []

        # Convert to numpy array if needed
        features = np.array(features, dtype=np.float32)

        # Check for NaNs
        nan_mask = np.isnan(features)
        rows_with_nan = np.any(nan_mask, axis=1)

        if self.missing_data_strategy == "skip":
            # Keep only rows with complete data
            keep_indices = np.where(~rows_with_nan)[0]
            cleaned_features = features[keep_indices]
            cleaned_identifiers = [head_identifiers[i] for i in keep_indices]
            skipped_heads = [head_identifiers[i] for i in np.where(rows_with_nan)[0]]

        elif self.missing_data_strategy == "zero":
            # Replace NaNs with zeros
            cleaned_features = np.nan_to_num(features, nan=0.0)
            cleaned_identifiers = head_identifiers

        elif self.missing_data_strategy == "nan":
            # Impute missing values instead of skipping
            imputer = SimpleImputer(strategy=self.imputation_strategy)
            cleaned_features = imputer.fit_transform(features)
            cleaned_identifiers = head_identifiers

        else:
            raise ValueError(f"Invalid missing_data_strategy: {self.missing_data_strategy}")

        return cleaned_features, cleaned_identifiers, skipped_heads

    def _validate_features(self, original_features, cleaned_features, skipped_heads):
        """
        Print a validation report for transparency.
        """
        total_heads = len(original_features)
        usable_heads = len(cleaned_features)
        skipped = len(skipped_heads)

        print("\n" + "=" * 50)
        print("FEATURE VALIDATION REPORT")
        print("=" * 50)
        print(f"  Total heads: {total_heads}")
        print(f"  Skipped heads: {skipped} ({skipped / total_heads * 100:.2f}%)")
        print(f"  Final usable heads: {usable_heads}")
        print(f"  Missing data strategy: {self.missing_data_strategy}")
        if self.missing_data_strategy == "nan":
            print(f"  Imputation strategy: {self.imputation_strategy}")
        print("=" * 50)

    def cluster_heads(self, features, head_identifiers):
        """
        Main entry point for clustering heads.
        Args:
            features: np.ndarray, shape (n_heads_total, n_features)
            head_identifiers: list of (seq_idx, layer_idx, head_idx)
        Returns:
            clusters: dict mapping cluster_id -> list of head identifiers
            kmeans: trained KMeans model
            extra_info: dict with skipped_heads, cleaned_features, cleaned_identifiers
        """
        # Handle missing data
        cleaned_features, cleaned_identifiers, skipped_heads = self._preprocess_features(features, head_identifiers)

        # Validate feature quality
        self._validate_features(features, cleaned_features, skipped_heads)

        if cleaned_features.shape[0] == 0:
            raise ValueError("No valid features available for clustering after missing data handling!")

        # Standardize features before clustering
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(cleaned_features)

        # Perform clustering
        kmeans = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=10
        )
        cluster_labels = kmeans.fit_predict(features_scaled)

        # Group heads by cluster
        clusters = defaultdict(list)
        for idx, label in enumerate(cluster_labels):
            clusters[label].append(cleaned_identifiers[idx])

        return clusters, kmeans, {
            "cleaned_features": cleaned_features,
            "cleaned_identifiers": cleaned_identifiers,
            "skipped_heads": skipped_heads,
            "missing_data_strategy": self.missing_data_strategy,
            "imputation_strategy": self.imputation_strategy if self.missing_data_strategy == "nan" else None
        }

    def interpret_clusters(self, clusters, kmeans):
        """
        Interpret cluster centers in terms of feature contributions.
        Returns dict: cluster_id -> interpretation
        """
        interpretations = {}
        centers = kmeans.cluster_centers_

        for cluster_id, center in enumerate(centers):
            feature_ranking = np.argsort(-np.abs(center))  # Rank by magnitude
            interpretations[cluster_id] = {
                "top_features_indices": feature_ranking[:5],
                "center_values": center,
                "num_heads": len(clusters[cluster_id])
            }

        return interpretations


# Stat pipeline 

In [ ]:
from collections import defaultdict
import numpy as np
from scipy.stats import spearmanr

class Phase1AnalysisPipeline:
    """
    Phase-1 analysis: compute mandatory stats for attention heads.
    Works directly with features extracted per head and sequence metadata.
    Outcome info is embedded in sequence_metadata['outcome'].
    """

    def __init__(self, features_array, head_identifiers, sequence_metadata):
        self.features_array = features_array  # shape: [num_heads, num_features]
        self.head_identifiers = head_identifiers  # list of (seq_idx, layer_idx, head_idx)
        self.sequence_metadata = sequence_metadata

    # -----------------------------------
    # Main entry
    # -----------------------------------
    def run_phase1(self, clusters):
        """
        Compute Phase-1 mandatory stats:
        - Layer specialization (mean, CI)
        - Complexity trends (n_shots)
        - Cluster-layer enrichment
        - Cluster-performance mapping
        """
        print("Computing Layer Specialization...")
        layer_stats = self.compute_layer_specialization(clusters)
        
        print("Computing Complexity Trends...")
        complexity_stats = self.compute_complexity_trends(clusters)
        
        print("Computing Cluster-Layer Enrichment...")
        cluster_layer_stats = self.compute_cluster_layer_enrichment(clusters)
        
        print("Computing Cluster-Performance Mapping...")
        cluster_perf_stats = self.compute_cluster_performance(clusters)
        
        return {
            'layer_specialization': layer_stats,
            'complexity_trends': complexity_stats,
            'cluster_layer_enrichment': cluster_layer_stats,
            'cluster_performance': cluster_perf_stats
        }

    # -----------------------------------
    # Layer specialization stats
    # -----------------------------------
    def compute_layer_specialization(self, clusters):
        """
        Compute mean attention features per layer and cluster.
        Returns dict: layer_idx -> cluster -> mean_feature_vector
        """
        layer_cluster_features = defaultdict(lambda: defaultdict(list))
        for head_idx, (seq_idx, layer_idx, head_id) in enumerate(self.head_identifiers):
            cluster_label = clusters[head_idx]
            layer_cluster_features[layer_idx][cluster_label].append(self.features_array[head_idx])

        # Compute means
        layer_stats = {}
        for layer_idx, cluster_dict in layer_cluster_features.items():
            layer_stats[layer_idx] = {}
            for cluster_label, feats in cluster_dict.items():
                layer_stats[layer_idx][cluster_label] = np.mean(feats, axis=0)
        return layer_stats

    # -----------------------------------
    # Complexity trends: n_shots
    # -----------------------------------
    def compute_complexity_trends(self, clusters):
        """
        Analyze complexity trend (feature vs n_shots)
        Returns dict: cluster -> {'slope', 'spearman_r', 'effect_size'}
        """
        cluster_trends = defaultdict(dict)
        for cluster_label in np.unique(clusters):
            feats = []
            n_shots = []
            for head_idx, (seq_idx, layer_idx, head_id) in enumerate(self.head_identifiers):
                if clusters[head_idx] != cluster_label:
                    continue
                feats.append(self.features_array[head_idx][0])  # e.g., entropy or main stat
                n_shots.append(self.sequence_metadata[seq_idx]['n_shots'])
            if len(feats) > 1:
                # Linear slope
                slope = np.polyfit(n_shots, feats, 1)[0]
                # Spearman correlation
                rho, _ = spearmanr(n_shots, feats)
                # Cohen's d between high vs low n_shots (simplified)
                low_feats = [f for f, n in zip(feats, n_shots) if n <= np.median(n_shots)]
                high_feats = [f for f, n in zip(feats, n_shots) if n > np.median(n_shots)]
                effect_size = (np.mean(high_feats) - np.mean(low_feats)) / np.sqrt(
                    0.5*(np.var(high_feats)+np.var(low_feats)+1e-8)
                )
                cluster_trends[cluster_label] = {
                    'slope': slope,
                    'spearman_r': rho,
                    'effect_size': effect_size
                }
        return cluster_trends

    # -----------------------------------
    # Cluster-Layer enrichment
    # -----------------------------------
    def compute_cluster_layer_enrichment(self, clusters):
        """
        Count heads per cluster per layer
        Returns dict: layer_idx -> cluster -> count
        """
        enrichment = defaultdict(lambda: defaultdict(int))
        for head_idx, (seq_idx, layer_idx, head_id) in enumerate(self.head_identifiers):
            cluster_label = clusters[head_idx]
            enrichment[layer_idx][cluster_label] += 1
        return enrichment

    # -----------------------------------
    # Cluster-performance mapping
    # -----------------------------------
    def compute_cluster_performance(self, clusters):
        """
        Map clusters to outcomes (Mem / ID / OOD) using sequence_metadata['outcome']
        Returns dict: cluster -> outcome -> list of feature vectors
        """
        cluster_perf = defaultdict(lambda: defaultdict(list))
        for head_idx, (seq_idx, layer_idx, head_id) in enumerate(self.head_identifiers):
            cluster_label = clusters[head_idx]
            outcome = self.sequence_metadata[seq_idx]['outcome']
            cluster_perf[cluster_label][outcome].append(self.features_array[head_idx])
        return cluster_perf


# Visualization

In [ ]:
def visualize_clusters(features, cluster_labels, interpretations):
    """Create visualizations of discovered clusters"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Plot 1: Entropy vs Locality
    axes[0,0].scatter(features[:, 0], features[:, 1], c=cluster_labels, cmap='viridis', alpha=0.6)
    axes[0,0].set_xlabel('Attention Entropy')
    axes[0,0].set_ylabel('Locality Score')
    axes[0,0].set_title('Entropy vs Locality')
    
    # Plot 2: Cross-example vs Context-to-query
    axes[0,1].scatter(features[:, 2], features[:, 3], c=cluster_labels, cmap='viridis', alpha=0.6)
    axes[0,1].set_xlabel('Cross-Example Attention')
    axes[0,1].set_ylabel('Context-to-Query Attention')
    axes[0,1].set_title('Cross-Example vs ICL Patterns')
    
    # Plot 3: Layer distribution
    for cluster_id in range(max(cluster_labels) + 1):
        cluster_mask = cluster_labels == cluster_id
        cluster_layers = features[cluster_mask, 6]  # Layer index
        axes[1,0].hist(cluster_layers, alpha=0.5, label=f'Cluster {cluster_id}', bins=6)
    axes[1,0].set_xlabel('Layer')
    axes[1,0].set_ylabel('Count')
    axes[1,0].set_title('Cluster Distribution Across Layers')
    axes[1,0].legend()
    
    # Plot 4: Summary table
    axes[1,1].axis('off')
    summary_text = "Cluster Interpretations:\n\n"
    for cluster_id, interp in interpretations.items():
        summary_text += f"Cluster {cluster_id}: {interp['type']}\n"
        summary_text += f"  Heads: {interp['n_heads']}\n\n"
    axes[1,1].text(0.1, 0.9, summary_text, transform=axes[1,1].transAxes, 
                   fontsize=10, verticalalignment='top')
    
    plt.tight_layout()
    return fig

def run_phase1_analysis(attention_data, sequence_metadata):
    """Main function to run Phase 1 analysis"""
    config = AnalysisConfig()
    extractor = AttentionPatternExtractor(config)
    clusterer = HeadClusterer(config)
    
    print("Extracting attention features...")
    features, head_identifiers = extractor.extract_head_features(attention_data, sequence_metadata)
    
    print("Clustering heads...")
    clusters, kmeans = clusterer.cluster_heads(features, head_identifiers)
    
    print("Interpreting clusters...")
    interpretations = clusterer.interpret_clusters(clusters, kmeans)
    
    print("Creating visualizations...")
    #fig = visualize_clusters(features, kmeans.labels_, interpretations)
    
    return {
        'clusters': clusters,
        'interpretations': interpretations,
        'features': features,
        'head_identifiers': head_identifiers,
        #'figure': fig
    }

# Run analysis

In [10]:
# =========================================
# Updated run_phase1_analysis
# =========================================

def run_phase1_analysis(config,attention_data, sequence_metadata, n_clusters=4):
    """
    End-to-end Phase-1 pipeline:
    1. Extract features per head
    2. Cluster heads
    3. Compute mandatory Phase-1 stats
    
    Note: outcome info is now stored in sequence_metadata['outcome'] per sequence.
    """

    # -------------------------------
    # 1) Feature extraction
    # -------------------------------
    extractor = AttentionPatternExtractor(config)
    print("Extracting attention features...")
    features_array, head_identifiers = extractor.extract_head_features(attention_data, sequence_metadata)
    print(f"Extracted features shape: {features_array.shape}")

    # -------------------------------
    # 2) Head clustering
    # -------------------------------
    clusterer = HeadClusterer(config)
    print(f"Clustering heads into {n_clusters} clusters...")
    clusters, kmeans_model = clusterer.cluster_heads(features_array, head_identifiers)
    print(f"Clusters assigned: {np.unique(clusters)}")

    # -------------------------------
    # 3) Phase-1 Statistical Analysis
    # -------------------------------
    print("Running Phase-1 statistics...")
    phase1_pipeline = Phase1AnalysisPipeline(
        features_array=features_array,
        head_identifiers=head_identifiers,
        sequence_metadata=sequence_metadata
    )

    phase1_stats = phase1_pipeline.run_phase1(clusters)

    # -------------------------------
    # 4) Return combined results
    # -------------------------------
    return {
        'features_array': features_array,
        'head_identifiers': head_identifiers,
        'clusters': clusters,
        'kmeans_model': kmeans_model,
        'phase1_stats': phase1_stats
    }


In [17]:
class Config:
    def __init__(self, n_layers=6, n_heads=8):
        self.n_layers = n_layers
        self.n_heads = n_heads

config = Config(n_layers=6, n_heads=8)

In [ ]:
data_dir = root_dir / "results/uniform_1000_L3_M3/clm_noshuffle_seedbalanced/memorization/collection/raw_evaluations/attention_data/clm_noshuffle_seedbalanced_step36880"

tokens_per_example = 9
loader = AttentionDataLoader(data_dir, tokens_per_example)
print("Loading attention files...")
attention_data, sequence_metadata, completeness_info = loader.load_attention_files()
attention_data, sequence_metadata = loader.convert_to_analysis_format(attention_data, sequence_metadata)


Loading attention files...
Found 57600 .npz files
Successfully loaded data for 200 sequences


In [19]:
data_dir = root_dir / "results/uniform_1000_L3_M3/clm_noshuffle_seedbalanced/memorization/collection/raw_evaluations/attention_data/clm_noshuffle_seedbalanced_step36880"
loader = AttentionDataLoaderBatch(data_dir, batch_size=50, tokens_per_example = 9)
pipeline = Phase1AnalysisPipelineBatch(config)
results = pipeline.run_phase1_batches(loader.batch_generator())


AttributeError: 'Config' object has no attribute 'n_clusters'

In [ ]:
def load_and_analyze(data_dir, step=None, max_sequences=100, tokens_per_example=3):
    """Complete pipeline: load data and run Phase 1 analysis"""
    # Load data
    loader = AttentionDataLoader(data_dir, tokens_per_example)
    print("Loading attention files...")
    files_data, sequence_info = loader.load_attention_files(step=step, max_sequences=max_sequences)
    
    # Convert to analysis format
    print("Converting to analysis format...")
    attention_data, sequence_metadata = loader.convert_to_analysis_format(files_data, sequence_info)
    
    # Validate data
    complete_sequences = loader.validate_data(attention_data, sequence_metadata)
    
    # if complete_sequences < 10:
    #     print("Warning: Very few complete sequences. Consider checking data or reducing n_layers/n_heads.")
    
    # Filter out incomplete sequences for analysis
    complete_attention_data = []
    complete_sequence_metadata = []
    
    for i, seq_attention in enumerate(attention_data):
        seq_complete = True
        for layer_idx in seq_attention:
            for head_idx in seq_attention[layer_idx]:
                if seq_attention[layer_idx][head_idx] is None:
                    seq_complete = False
                    break
            if not seq_complete:
                break
        
        if seq_complete:
            complete_attention_data.append(seq_attention)
            complete_sequence_metadata.append(sequence_metadata[i])
    
    print(f"Using {len(complete_attention_data)} complete sequences for analysis")
    
    # Run Phase 1 analysis
    if complete_attention_data:
        print("Running Phase 1 analysis...")
        results = run_phase1_analysis(complete_attention_data, complete_sequence_metadata)
        return results, loader
    else:
        print("No complete sequences available for analysis")
        return None, loader


In [14]:
results = run_phase1_analysis(config,attention_data, sequence_metadata, n_clusters=4)

Extracting attention features...


: 

# old code

In [ ]:
def load_and_analyze(data_dir, step=None, max_sequences=100, tokens_per_example=3):
    """Complete pipeline: load data and run Phase 1 analysis"""
    # Load data
    loader = AttentionDataLoader(data_dir, tokens_per_example)
    print("Loading attention files...")
    files_data, sequence_info = loader.load_attention_files(step=step, max_sequences=max_sequences)
    
    # Convert to analysis format
    print("Converting to analysis format...")
    attention_data, sequence_metadata = loader.convert_to_analysis_format(files_data, sequence_info)
    
    # Validate data
    complete_sequences = loader.validate_data(attention_data, sequence_metadata)
    
    # if complete_sequences < 10:
    #     print("Warning: Very few complete sequences. Consider checking data or reducing n_layers/n_heads.")
    
    # Filter out incomplete sequences for analysis
    complete_attention_data = []
    complete_sequence_metadata = []
    
    for i, seq_attention in enumerate(attention_data):
        seq_complete = True
        for layer_idx in seq_attention:
            for head_idx in seq_attention[layer_idx]:
                if seq_attention[layer_idx][head_idx] is None:
                    seq_complete = False
                    break
            if not seq_complete:
                break
        
        if seq_complete:
            complete_attention_data.append(seq_attention)
            complete_sequence_metadata.append(sequence_metadata[i])
    
    print(f"Using {len(complete_attention_data)} complete sequences for analysis")
    
    # Run Phase 1 analysis
    if complete_attention_data:
        print("Running Phase 1 analysis...")
        results = run_phase1_analysis(complete_attention_data, complete_sequence_metadata)
        return results, loader
    else:
        print("No complete sequences available for analysis")
        return None, loader


In [33]:
data_dir = root_dir / "results/uniform_1000_L3_M3/clm_noshuffle_seedbalanced/memorization/collection/raw_evaluations/attention_data/clm_noshuffle_seedbalanced_step36880"
tokens_per_example = 9
loader = AttentionDataLoader(data_dir, tokens_per_example)
print("Loading attention files...")
files_data, sequence_info = loader.load_attention_files()

# Convert to analysis format
print("Converting to analysis format...")
attention_data, sequence_metadata = loader.convert_to_analysis_format(files_data, sequence_info)

# Validate data
complete_sequences, missing_data = loader.validate_data(attention_data, sequence_metadata)


Loading attention files...
Found 2 .npz files
Converting to analysis format...
Data Validation Summary:
- Total sequences: 1
- Complete sequences: 1
- Total missing head/layer combinations: 0
- Shots distribution: {5: 1}


In [36]:
complete_attention_data = []
complete_sequence_metadata = []

for i, seq_attention in enumerate(attention_data):
    seq_complete = True
    for layer_idx in seq_attention:
        for head_idx in seq_attention[layer_idx]:
            if seq_attention[layer_idx][head_idx] is None:
                seq_complete = False
                break
        if not seq_complete:
            break
    
    if seq_complete:
        complete_attention_data.append(seq_attention)
        complete_sequence_metadata.append(sequence_metadata[i])

print(f"Using {len(complete_attention_data)} complete sequences for analysis")


Using 1 complete sequences for analysis


In [ ]:
import numpy as np
from pathlib import Path
from collections import defaultdict
import re
from scipy.stats import entropy

# Configuration class for the analysis
class Config:
    def __init__(self, n_layers=6, n_heads=8):
        self.n_layers = n_layers
        self.n_heads = n_heads

# Example usage of the revised AttentionDataLoader and AttentionPatternExtractor
def run_attention_analysis(data_dir, config):
    """Complete example of running attention analysis with robust data handling"""
    
    print("="*60)
    print("ATTENTION PATTERN ANALYSIS")
    print("="*60)
    
    # Initialize data loader
    loader = AttentionDataLoader(
        data_dir=data_dir,
        tokens_per_example=3,
        n_layers=config.n_layers,
        n_heads=config.n_heads
    )
    
    # Example 1: Load complete sequences only from specific step
    print("\n1. Loading complete sequences from step 36880...")
    files_data, sequence_info, completeness_info = loader.load_attention_files(
        step=36880,
        require_complete=True,
        max_sequences=50,
        sampling_strategy='stratified'
    )
    
    # Example 2: Load specific sequences with certain shot counts
    print("\n2. Loading sequences with 3 or 5 shots...")
    files_data_filtered, sequence_info_filtered, completeness_filtered = loader.load_attention_files(
        n_shots_filter=[3, 5],
        require_complete=False,  # Allow incomplete sequences
        max_sequences=30,
        sampling_strategy='random'
    )
    
    # Example 3: Load specific sequence ID range
    print("\n3. Loading sequence IDs 10-20...")
    files_data_range, sequence_info_range, completeness_range = loader.load_attention_files(
        seq_id_range=(10, 20),
        require_complete=True
    )
    
    # Convert to analysis format
    print("\n4. Converting to analysis format...")
    attention_data, sequence_metadata = loader.convert_to_analysis_format(
        files_data, sequence_info
    )
    
    # Validate loaded data
    print("\n5. Validating loaded data...")
    complete_count = loader.validate_data(attention_data, sequence_metadata)
    
    # Get detailed statistics
    stats = loader.get_data_statistics(completeness_info)
    print(f"\nData Statistics:")
    print(f"- Completion rate: {stats['completion_rate']*100:.1f}%")
    print(f"- Complete sequences: {stats['complete_sequences']}")
    
    # Initialize pattern extractor with different missing data strategies
    print("\n6. Analyzing attention patterns...")
    
    # Strategy 1: Skip missing data
    extractor_skip = AttentionPatternExtractor(config, missing_data_strategy='skip')
    extractor_skip.validate_attention_data(attention_data, sequence_metadata)
    features_skip, identifiers_skip = extractor_skip.extract_head_features(
        attention_data, sequence_metadata
    )
    print(f"Features extracted (skip strategy): {features_skip.shape}")
    
    # Strategy 2: Use NaN for missing data
    extractor_nan = AttentionPatternExtractor(config, missing_data_strategy='nan')
    features_nan, identifiers_nan = extractor_nan.extract_head_features(
        attention_data, sequence_metadata
    )
    print(f"Features extracted (nan strategy): {features_nan.shape}")
    
    # Strategy 3: Use zero for missing data
    extractor_zero = AttentionPatternExtractor(config, missing_data_strategy='zero')
    features_zero, identifiers_zero = extractor_zero.extract_head_features(
        attention_data, sequence_metadata
    )
    print(f"Features extracted (zero strategy): {features_zero.shape}")
    
    # Analyze feature quality
    print("\n7. Feature Quality Analysis:")
    
    feature_names = ['entropy', 'locality', 'cross_example', 'context_to_query', 
                    'query_to_context', 'rule_complexity', 'layer', 'head']
    
    for strategy, features in [('skip', features_skip), ('nan', features_nan), ('zero', features_zero)]:
        print(f"\n{strategy.upper()} Strategy:")
        if len(features) > 0:
            for i, name in enumerate(feature_names):
                col = features[:, i]
                if name in ['layer', 'head', 'rule_complexity']:
                    print(f"  {name}: range [{col.min():.0f}, {col.max():.0f}]")
                else:
                    valid_mask = ~np.isnan(col) if strategy == 'nan' else np.ones(len(col), dtype=bool)
                    if valid_mask.sum() > 0:
                        valid_col = col[valid_mask]
                        print(f"  {name}: mean={valid_col.mean():.4f}, std={valid_col.std():.4f}")
                    else:
                        print(f"  {name}: all NaN")
        else:
            print("  No features extracted")
    
    return {
        'loader': loader,
        'extractors': {
            'skip': extractor_skip,
            'nan': extractor_nan, 
            'zero': extractor_zero
        },
        'features': {
            'skip': (features_skip, identifiers_skip),
            'nan': (features_nan, identifiers_nan),
            'zero': (features_zero, identifiers_zero)
        },
        'data': (attention_data, sequence_metadata),
        'stats': stats
    }


In [54]:
config = Config(n_layers=6, n_heads=8)
loader = AttentionDataLoader(data_dir, n_layers=6, n_heads=8, tokens_per_example=9)
    

In [55]:
files_data, sequence_info, completeness = loader.load_attention_files(
        require_complete=False
    )

Found 2 .npz files
Successfully loaded data for 1 sequences
Complete sequences: 0
Incomplete sequences: 1
Incomplete sequence details:
  Seq 9: 4.2% coverage, missing 46 combinations


In [56]:
sequence_info

{9: {'n_shots': 5, 'step': 36880}}

In [57]:
attention_data, sequence_metadata = loader.convert_to_analysis_format(
        files_data, sequence_info
    )

In [59]:
extractor = AttentionPatternExtractor(config, missing_data_strategy='skip')
features, identifiers = extractor.extract_head_features(
        attention_data, sequence_metadata
    )
    

Skipped 46 head features due to missing data


In [60]:
complexity_groups = defaultdict(list)
for i, identifier in enumerate(identifiers):
        seq_idx, layer_idx, head_idx = identifier
        complexity = sequence_metadata[seq_idx]['rule_complexity']
        complexity_groups[complexity].append(i)
    
print(f"\nFeatures by complexity:")
for complexity, indices in complexity_groups.items():
        complexity_features = features[indices]
        print(f"  Complexity {complexity}: {len(indices)} features")
        print(f"    Mean entropy: {complexity_features[:, 0].mean():.4f}")
        print(f"    Mean locality: {complexity_features[:, 1].mean():.4f}")
    


Features by complexity:
  Complexity 3: 2 features
    Mean entropy: 2.8132
    Mean locality: 0.4521


In [63]:
complexity_features.shape

(2, 8)

In [ ]:

# Example of advanced filtering and analysis
def advanced_analysis_example(data_dir):
    """Example showing advanced filtering and analysis capabilities"""
    
    config = Config(n_layers=6, n_heads=8)
    loader = AttentionDataLoader(data_dir, n_layers=6, n_heads=8)
    
    # Load data with multiple filtering criteria
    print("Advanced filtering example:")
    
    # Get sequences from steps 30000-40000, with 3-5 shots, sequence IDs 1-100
    files_data, sequence_info, completeness = loader.load_attention_files(
        step_range=(30000, 40000),
        n_shots_filter=[3, 4, 5],
        seq_id_range=(1, 100),
        max_sequences=20,
        sampling_strategy='stratified',
        require_complete=True
    )
    
    print(f"Loaded {len(files_data)} sequences with advanced filtering")
    
    # Convert and analyze
    attention_data, sequence_metadata = loader.convert_to_analysis_format(
        files_data, sequence_info
    )
    
    # Analyze with robust extractor
    extractor = AttentionPatternExtractor(config, missing_data_strategy='skip')
    features, identifiers = extractor.extract_head_features(
        attention_data, sequence_metadata
    )
    
    # Group by complexity for analysis
    complexity_groups = defaultdict(list)
    for i, identifier in enumerate(identifiers):
        seq_idx, layer_idx, head_idx = identifier
        complexity = sequence_metadata[seq_idx]['rule_complexity']
        complexity_groups[complexity].append(i)
    
    print(f"\nFeatures by complexity:")
    for complexity, indices in complexity_groups.items():
        complexity_features = features[indices]
        print(f"  Complexity {complexity}: {len(indices)} features")
        print(f"    Mean entropy: {complexity_features[:, 0].mean():.4f}")
        print(f"    Mean locality: {complexity_features[:, 1].mean():.4f}")
    
    return features, identifiers, sequence_metadata

# Main execution example
if __name__ == "__main__":
    # Example usage
    data_directory = "/path/to/attention/data"
    config = Config(n_layers=6, n_heads=8)
    
    # Run complete analysis
    results = run_attention_analysis(data_directory, config)
    
    # Run advanced analysis
    # advanced_features, advanced_ids, advanced_metadata = advanced_analysis_example(data_directory)